# Landing → Bronze | CineData Analytics

Este notebook faz a **ingestão da camada Bronze** (Arquitetura Medalhão):

- Lê os 5 CSVs do Volume (`landing`) **sem alterar estrutura nem conteúdo**.
- Adiciona a coluna `ingestion_datetime` (momento da inserção na Bronze).
- Grava em **Delta** no modo **append**.
- Extrai a cotação do dólar (PTAX) da API do Banco Central para `bronze.tb_cotacao_dolar`.

## 1. Configurações e imports

In [0]:
from datetime import date, timedelta

import requests
from pyspark.sql import functions as F

CATALOGO = "workspace"
BANCO_BRONZE = "bronze"

# Pasta do Volume
CAMINHO_INPUTS = "/Volumes/workspace/default/landing/Inputs - Atividade Engenharia de Dados - Bases de Dados"

# Mapeamento: arquivo original -> tabela Bronze
# O nome do arquivo disponível no Volume é: movies_info_TMDB_IMDB.csv
# Embora o enunciado apresente movies_info_IMDB_TMDB.csv.
ARQUIVOS_BRONZE = {
    "movies_info_TMDB_IMDB.csv":           "tb_movies_info",
    "movies_financials_IMDB_TMDB.csv":     "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv":        "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv":      "tb_credits_and_tags",
    "movies_reviews.csv":                  "tb_movies_reviews",
}

for arquivo in ARQUIVOS_BRONZE:
    caminho = f"{CAMINHO_INPUTS}/{arquivo}"
    try:
        dbutils.fs.ls(caminho)
    except Exception:
        raise FileNotFoundError(
            f"Arquivo obrigatório não encontrado: {arquivo}"
        )

## 2. Criação do banco de dados Bronze

In [0]:
spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BANCO_BRONZE}")

DataFrame[]

## 3. Ingestão dos CSVs

Decisões e justificativas:

- **`inferSchema = False`**: todas as colunas entram como `STRING`. A Bronze não pode arrumar nada;
  se o Spark inferisse tipos, valores sujos poderiam virar `NULL` e perderíamos o dado original. A tipagem correta é responsabilidade da Silver.
- **`multiLine = True` + `quote`/`escape`**: sinopses (overview) podem ter vírgulas, aspas e quebras de linha
  dentro do texto. Sem isso, uma sinopse quebraria em várias linhas e desalinharia as colunas.
- **`current_timestamp()`**: gera o `ingestion_datetime` no momento da gravação. A Silver usa essa coluna
  para escolher a versão mais recente quando houver duplicados.
- **Modo append**: exigido no enunciado. Rodar o notebook mais de uma vez insere as linhas de novo.

In [0]:
def ingerir_csv(nome_arquivo: str, nome_tabela: str) -> None:
    """Lê um CSV do Volume e grava na Bronze sem alterar o conteúdo original."""
    caminho = f"{CAMINHO_INPUTS}/{nome_arquivo}"

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)   # tudo como string: preserva o dado bruto
        .option("multiLine", True)      # textos longos com quebra de linha
        .option("quote", '"')
        .option("escape", '"')
        .csv(caminho)
    )

    # Única coluna adicionada: timestamp da inserção na Bronze
    df = df.withColumn("ingestion_datetime", F.current_timestamp())

    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(f"{BANCO_BRONZE}.{nome_tabela}")
    )
    print(f"OK  {BANCO_BRONZE}.{nome_tabela}  <-  {nome_arquivo}")


for arquivo, tabela in ARQUIVOS_BRONZE.items():
    ingerir_csv(arquivo, tabela)

OK  bronze.tb_movies_info  <-  movies_info_TMDB_IMDB.csv
OK  bronze.tb_movies_financials  <-  movies_financials_IMDB_TMDB.csv
OK  bronze.tb_movies_metrics  <-  movies_metrics_IMDB_TMDB.csv
OK  bronze.tb_credits_and_tags  <-  credits_and_tags_IMDB_TMDB.csv
OK  bronze.tb_movies_reviews  <-  movies_reviews.csv


## 4. Ingestão da API do Banco Central (cotação do dólar)

- Datas por **widgets**, no formato **MM-DD-AAAA** exigido pela API.
- Se os widgets ficarem vazios (caso do Job agendado), usamos os **últimos 7 dias corridos** a partir da data
  de execução. A janela de 7 dias existe porque a API não retorna cotação em finais de semana e feriados.
- Os dados da API são gravados como retornaram (`dataHoraCotacao` e `cotacaoCompra`), mais o `ingestion_datetime`.

In [0]:
dbutils.widgets.text("data_inicio", "", "Data início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", "", "Data fim (MM-DD-AAAA)")

hoje = date.today()
data_inicio = dbutils.widgets.get("data_inicio").strip() or (hoje - timedelta(days=7)).strftime("%m-%d-%Y")
data_fim = dbutils.widgets.get("data_fim").strip() or hoje.strftime("%m-%d-%Y")

if data_inicio > data_fim:
    raise ValueError("A data inicial não pode ser maior que a data final.")

print(f"Consultando cotação de {data_inicio} até {data_fim}")

Consultando cotação de 09-14-2026 até 09-21-2026


In [0]:
url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

resposta = requests.get(url, timeout=30)
resposta.raise_for_status()   # falha o notebook (e o Job) se a API retornar erro HTTP

registros = resposta.json().get("value", [])
print(f"{len(registros)} cotações retornadas pela API")

# Schema: funciona mesmo se a API devolver lista vazia
df_cotacao = spark.createDataFrame(
    [(r["dataHoraCotacao"], r["cotacaoCompra"]) for r in registros],
    schema="dataHoraCotacao string, cotacaoCompra double",
).withColumn("ingestion_datetime", F.current_timestamp())

(
    df_cotacao.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{BANCO_BRONZE}.tb_cotacao_dolar")
)
print(f"OK  {BANCO_BRONZE}.tb_cotacao_dolar")

6 cotações retornadas pela API
OK  bronze.tb_cotacao_dolar


## 5. Validação e inspeção

Contagem de linhas e schema de cada tabela Bronze.

In [0]:
todas_tabelas = list(ARQUIVOS_BRONZE.values()) + ["tb_cotacao_dolar"]

for tabela in todas_tabelas:
    df = spark.table(f"{BANCO_BRONZE}.{tabela}")
    print(f"\n=== {BANCO_BRONZE}.{tabela} | {df.count()} linhas ===")
    df.printSchema()


=== bronze.tb_movies_info | 213192 linhas ===
root
 |-- id: string (nullable = true)
 |-- tconst: string (nullable = true)
 |-- title: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- original_language: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- runtime: string (nullable = true)
 |-- status: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- tagline: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)


=== bronze.tb_movies_financials | 212330 linhas ===
root
 |-- id: string (nullable = true)
 |-- budget: string (nullable = true)
 |-- revenue: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)


=== bronze.tb_movies_metrics | 206144 linhas ===
root
 |-- id: string (nullable = true)
 |-- popularity: string (nullable = true)
 |-- vote_average: string (nullable = true)
 |-- vote_count: string (nullable = true)
 |-- averageRating: string (nullable = true)
 |-- 

In [0]:
# Amostra de cada tabela (útil para conhecer a sujeira que a Silver vai tratar)
for tabela in todas_tabelas:
    print(f"--- {tabela} ---")
    display(spark.table(f"{BANCO_BRONZE}.{tabela}").limit(5))

--- tb_movies_info ---


id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_datetime
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-09-18T20:10:17.204Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-09-18T20:10:17.204Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-09-18T20:10:17.204Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-09-18T20:10:17.204Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-09-18T20:10:17.204Z


--- tb_movies_financials ---


id,budget,revenue,ingestion_datetime
293660,58000000,Unknown,2026-09-18T20:10:26.774Z
299536,300000000,2052415039,2026-09-18T20:10:26.774Z
299534,356000000,2800000000,2026-09-18T20:10:26.774Z
475557,55000000,1074458282,2026-09-18T20:10:26.774Z
271110,250000000,Não Informado,2026-09-18T20:10:26.774Z


--- tb_movies_metrics ---


...

id,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,cast,ingestion_datetime
293660,"Action, Adventure, Comedy","20th Century Fox, The Donners' Company, Genre Films",United States of America,English,"superhero, anti hero, mercenary, based on comic, aftercreditsstinger, duringcreditsstinger",Tim Miller,"Rhett Reese, Paul Wernick","Ryan Reynolds, Morena Baccarin, Ed Skrein, T.J. Miller, Gina Carano, Leslie Uggams, Brianna Hildebrand, Stefan Kapičić, Karan Soni, Randal Reeder",2026-09-18T20:10:36.132Z
299536,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"English, Xhosa","sacrifice, magic, superhero, based on comic, space, battlefield, genocide, magical object, super power, aftercreditsstinger, marvel cinematic universe (mcu), cosmic","Anthony Russo, Joe Russo",N/A,"Robert Downey Jr., Chris Evans, Chris Hemsworth, Josh Brolin, Mark Ruffalo, Scarlett Johansson, Don Cheadle, Benedict Cumberbatch, Tom Holland, Chadwick Boseman",2026-09-18T20:10:36.132Z
299534,"Adventure, Science Fiction, Action",Marvel Studios,United States of America,"English, Japanese, Xhosa","superhero, time travel, space travel, time machine, based on comic, sequel, alien invasion, superhero team, marvel cinematic universe (mcu), alternate timeline, father daughter relationship, sister sister relationship","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Stan Lee, Jack Kirby, Joe Simon, Steve Englehart, Steve Gan, Bill Mantlo, Keith Giffen, Jim Starlin, Larry Lieber, Don Heck","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner, Josh Brolin, Don Cheadle, Paul Rudd, Benedict Cumberbatch",2026-09-18T20:10:36.132Z
475557,"Crime, Thriller, Drama","Warner Bros. Pictures, Joint Effort, Village Roadshow Pictures, Bron Studios, DC Films","Canada, United States of America",English,"dream, street gang, society, psychopath, clown, villain, based on comic, murder, psychological thriller, criminal mastermind, mental illness, anarchy, character study, clown makeup, subway train, social realism, supervillain, tv host, 1980s, mother son relationship, origin story, falling into madness, depressing",Todd Phillips,"Todd Phillips, Scott Silver, Bob Kane, Bill Finger, Jerry Robinson","Joaquin Phoenix, Robert De Niro, Zazie Beetz, Frances Conroy, Brett Cullen, Shea Whigham, Bill Camp, Glenn Fleshler, Leigh Gill, Josh Pais",2026-09-18T20:10:36.132Z
271110,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"Romanian, English, German, Russian","civil war, superhero, based on comic, sequel, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu), excited","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Joe Simon, Jack Kirby","Chris Evans, Robert Downey Jr., Scarlett Johansson, Sebastian Stan, Anthony Mackie, Don Cheadle, Jeremy Renner, Chadwick Boseman, Paul Bettany, Elizabeth Olsen",2026-09-18T20:10:36.132Z


--- tb_movies_reviews ---


id,nome,nota,comentario,ingestion_datetime
442113,Mariana Cardoso 277,4.4,null,2026-09-21T18:42:51.361Z
637007,Lucas Reis 602,3.9,null,2026-09-21T18:42:51.361Z
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.,2026-09-21T18:42:51.361Z
413036,Gabriela Monteiro 401,7.5,null,2026-09-21T18:42:51.361Z
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.,2026-09-21T18:42:51.361Z


--- tb_cotacao_dolar ---


dataHoraCotacao,cotacaoCompra,ingestion_datetime
2026-09-11 13:07:22.532196,5.0912,2026-09-18T20:11:26.496Z
2026-09-14 13:10:08.144425,5.169,2026-09-18T20:11:26.496Z
2026-09-15 13:09:19.199664,5.1484,2026-09-18T20:11:26.496Z
2026-09-16 13:05:30.35873,5.152,2026-09-18T20:11:26.496Z
2026-09-17 13:03:21.858212,5.1515,2026-09-18T20:11:26.496Z
